# Install the engine and open a committed artifact

- **Level:** 1, Fundamentals
- **Prerequisites:** none, this is the entry notebook
- **Reading time:** about 20 minutes

## Objectives

By the end of this notebook you can:

- install the validation engine `riskvalidation` and say which version you are running;
- list the engine's catalogue of validation tests and read what one entry tells you;
- open the three kinds of document Contraste commits (the index, a case manifest and a case artifact) one key at a time;
- say where every later number of these notebooks comes from, and which engine version baked it.

## Setup

The notebook environment is pinned in `notebooks/requirements.txt`. Install it once, in a virtual environment, with the familiar command:

```bash
pip install -r notebooks/requirements.txt
```

or install only the package this notebook uses, with the same exact pin:

```bash
pip install riskvalidation==1.0.1
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` on Windows) builds the whole environment, `.venv-notebooks`, for you. A notebook never installs anything by itself.

The next cell prints the version of the installed package, so that you can compare it with the pin.

In [1]:
# Print the version of the package this notebook uses, to compare it with notebooks/requirements.txt
import importlib.metadata as md
import json
from collections import Counter
from pathlib import Path

import riskvalidation as rv

print("riskvalidation", md.version("riskvalidation"))
print("module version:", rv.__version__)

riskvalidation 1.0.1
module version: 1.0.1


## Data

Contraste is a validator's workbench: it fits models on public data, runs the engine's tests on them and commits the result as documents. This notebook opens those documents; it fits and tests nothing yet.

- **Owner:** this repository, https://github.com/fsantibanezleal/CAOS_Contraste. The documents are written by the offline pipeline (`data-pipeline/`) and the engine, and read here as they are committed.
- **The three kinds of document** (contract 2): the **index** `data/derived/manifests/index.json` lists the cases; a **manifest** `data/derived/manifests/<case>.json` describes one case and lists its artifacts; an **artifact** `data/derived/<case>/<variant>.json` holds the models, the test rows, the impact and the findings of one variant of the case.
- **The case we open, C01:** retail credit-card PD. Its real data is the UCI Taiwan credit-card file (Yeh, 2009, 30,000 cards), licence class `mirror-allowed`, CC BY 4.0, https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients. The second case we peek at, C22, is synthetic: it measures the tests on generators with a known truth.
- **Files read:** `data/derived/manifests/index.json`, `data/derived/manifests/C01.json`, `data/derived/C01/holdout.json` (and, for the provenance of step 5, the nine other documents the manifest lists) and `data/derived/C22/null.json`. Every path is relative to the repository root, and no raw data is needed.

The next cell finds the repository root (the notebook sits three folders below it) and defines a reader that stops with the file's path when a file is missing.

In [2]:
# The notebook sits three folders below the repository root; a missing file stops the notebook and names its path
ROOT = Path("../../..").resolve()
DERIVED = ROOT / "data" / "derived"


def read_json(relative):
    return json.loads((DERIVED / relative).read_text(encoding="utf-8"))


print("derived folder found:", DERIVED.is_dir())

derived folder found: True


## Method

### Step 1: the catalogue of tests

The engine registers every validation test with its identifier, family, null hypothesis (H0), primary source and the kind of decision it supports. `catalogue()` returns the registry, so a validator can ask what exists before running anything. The next cell counts the tests by family and shows one entry in full. The identifiers, the keys of the documents and the engine's own texts (a null hypothesis, a reference, a note) are printed as the engine writes them, in English, in both trees.

In [3]:
# List the registry and print one entry in full
from riskvalidation import catalogue

tests = catalogue()
print("registered tests:", len(tests))
for family, count in sorted(Counter(t.family for t in tests).items()):
    print(f"{family:16s} {count}")

entry = next(t for t in tests if t.test_id == "pd.jeffreys")
print()
print("test_id:", entry.test_id, "| family:", entry.family, "| decision:", entry.decision)
print("name:", entry.name)
print("H0:", entry.h0)
print("reference:", entry.reference)

registered tests: 134
aggregation      8
ccr              8
discrimination   4
fairness         12
governance       5
ifrs9            9
irrbb            7
lgd_ead          6
liquidity        11
market           12
ml               13
oprisk           5
pd-calibration   11
portfolio        5
rating-system    3
stability        5
stress           6
transitions      4

test_id: pd.jeffreys | family: pd-calibration | decision: p_value
name: Jeffreys test
H0: the PD applied at the start of the period is greater than or equal to the true PD
reference: ECB (2019), Instructions for reporting the validation results of internal models: IRB Pillar I models for credit risk, section 2.5.3.1; Brown, Cai and DasGupta (2001), doi:10.1214/ss/1009213286


### Step 2: the index

The index is the table of contents of the committed documents: how many cases there are, which one the site opens by default, and for each case its title, category and the path of its manifest. The next cell reads it and groups the cases by category.

In [4]:
# Read the index and group the cases by category and kind
index = read_json("manifests/index.json")
print("schema:", index["schema"])
print("cases:", index["n_cases"], "| default case:", index["default_case"])
print("kinds:", dict(Counter(c["kind"] for c in index["cases"])))
print("categories:", dict(Counter(c["category_id"] for c in index["cases"])))
print("index engine_version equals the VERSION file (the product release, not the riskvalidation version):",
      index["engine_version"] == (ROOT / "VERSION").read_text(encoding="utf-8").strip())
for case in index["cases"][:4]:
    print(case["case_id"], "|", case["category"]["en"], "|", case["title"]["en"], "|", case["title"]["es"])

schema: contraste.index/v1
cases: 22 | default case: C01
kinds: {'real': 15, 'synthetic': 7}
categories: {'credit-scoring': 3, 'ratings-calibration': 2, 'provisions': 2, 'lgd-ead': 2, 'portfolio-capital': 1, 'market-ccr': 3, 'balance-sheet': 3, 'operational': 1, 'capital-stress': 2, 'ml-ai-risk': 2, 'validator': 1}
index engine_version equals the VERSION file (the product release, not the riskvalidation version): True
C01 | Credit scoring | Retail cards PD: champion vs challenger | PD de tarjetas: campeón vs retador
C02 | Credit scoring | Corporate PD across 1 to 5 year horizons | PD corporativa a horizontes de 1 a 5 años
C03 | Credit scoring | SME PD by vintage through 2008 | PD de pymes por cosecha a través de 2008
C04 | Ratings and calibration | Rating transitions and TTC PD by grade | Transiciones de calificación y PD TTC por grado


### Step 3: the manifest of a case

A manifest says what a case asks, which sources it reads (with their licence class and attribution), the seed, the engine versions that produced it and the artifacts it lists with their sizes. It also carries the contract-1 report: how many input records were accepted, flagged or rejected. The next cell reads the manifest of C01 and prints the fields a reader needs first.

In [5]:
# Read the manifest of C01 and print its question, sources, seed and artifacts
manifest = read_json("manifests/C01.json")
print("keys:", list(manifest))
print("question:", manifest["question"]["en"])
print("seed:", manifest["seed"], "| default variant:", manifest["default_variant"])
for source in manifest["sources"]:
    detail = manifest["source_details"][source]
    print(source, "| class:", detail["class"], "| attribution:", detail["attribution"])
print("contract 1 on the Taiwan file:", manifest["contract"]["uci-taiwan"]["counts"])
for item in manifest["artifacts"]:
    print(f'{item["role"]:8s} {item["variant_id"]:18s} {item["path"]:28s} {item["bytes"]:>8d} bytes')

keys: ['schema', 'case_id', 'title', 'category', 'question', 'sources', 'source_details', 'seed', 'engine', 'default_variant', 'artifacts', 'contract', 'expect', 'depends_on']
question: Does a monotone WoE scorecard match the challengers the literature says can beat it, once calibration is required, and what do the validation tests say when the population shifts?
seed: 42 | default variant: holdout
uci-taiwan | class: mirror-allowed | attribution: Yeh, I-C. (2009). Default of Credit Card Clients. UCI Machine Learning Repository. https://doi.org/10.24432/C55S3H
uci-german | class: mirror-allowed | attribution: Hofmann, H. (1994). Statlog (German Credit Data). UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77
contract 1 on the Taiwan file: {'input': 30000, 'accepted': 30000, 'rejected': 0, 'flagged': 26284, 'excluded': 0}
models   models-taiwan      C01/models-taiwan.json         122630 bytes
models   models-german      C01/models-german.json          48495 bytes
models   

### Step 4: an artifact

An artifact is where the numbers live. Its envelope has the same keys for every case: the `model` records (the rungs of the ladder), the `outputs` the web draws, the `tests` (one row per engine test call, each a `TestResult` written as a dictionary), the `impact`, the `findings` a validator would write, the `provenance` and the `lane`. The next cell opens the holdout artifact of C01 and counts what is inside.

In [6]:
# Open the holdout artifact of C01 and count its parts
holdout = read_json("C01/holdout.json")
print("keys:", list(holdout))
print("schema:", holdout["schema"], "| case:", holdout["case_id"], "| variant:", holdout["variant_id"])
print("rungs:", [rung["rung"] for rung in holdout["model"]])
print("held-out cards:", holdout["outputs"]["n"], "| defaults:", holdout["outputs"]["defaults"])
print("test rows:", len(holdout["tests"]), "| findings:", len(holdout["findings"]))
print("lights:", dict(Counter(row["light"] for row in holdout["tests"])))
print("keys of a test row:", list(holdout["tests"][0]))
print("impact keys:", list(holdout["impact"]))

keys: ['schema', 'case_id', 'variant_id', 'model', 'outputs', 'tests', 'impact', 'findings', 'provenance', 'lane']
schema: contraste.case/v1 | case: C01 | variant: holdout
rungs: ['P0', 'P0', 'P1', 'P2', 'P2', 'P3', 'P4', 'P4']
held-out cards: 9000 | defaults: 1991
test rows: 187 | findings: 4
lights: {'not_evaluated': 31, 'green': 136, 'red': 12, 'amber': 8}
keys of a test row: ['test_id', 'model_id', 'segment', 'statistic', 'p_value', 'metric', 'h0', 'alternative', 'policy_version', 'alpha_amber', 'alpha_red', 'light', 'n', 'n_events', 'inputs_hash', 'reference', 'notes', 'extras']
impact keys: ['approval_rate', 'bad_rate_champion', 'bad_rate_challenger', 'el_champion', 'el_challenger', 'loss_champion', 'loss_challenger', 'swap_in', 'swap_out', 'swap_in_bad_rate', 'swap_out_bad_rate', 'capital_champion', 'capital_challenger']


### Step 5: provenance and the engine that baked it

Every artifact says how it was made: the truth status of its data (`real-outcomes` or `synthetic-known-truth`), the seed, and the `riskvalidation` version that computed its tests. The committed documents were baked at different times with different engine releases, and the notebook runs the release you installed. So whenever a later notebook recomputes a committed number, it prints the baked version beside the installed one; a committed document is never rewritten by a notebook. The next cell compares them for every document of C01.

In [7]:
# Compare the engine version that baked each document of C01 with the installed one
provenance = holdout["provenance"]
print("truth status:", provenance["truth_status"], "| seed:", provenance["seed"])
print("installed riskvalidation:", rv.__version__)

baked = Counter(read_json(item["path"])["provenance"]["riskvalidation_version"] for item in manifest["artifacts"])
print("riskvalidation versions that baked the documents of C01:", dict(baked))
print("documents baked with the installed version:", baked.get(rv.__version__, 0), "of", len(manifest["artifacts"]))

truth status: real-outcomes | seed: 42
installed riskvalidation: 1.0.1
riskvalidation versions that baked the documents of C01: {'1.0.1': 10}
documents baked with the installed version: 10 of 10


### Step 6: a second kind of artifact

C22 measures the validation tests themselves, on generators whose truth is known. Its artifact has the same envelope, but the content of `outputs` differs and its data is synthetic by construction. The next cell opens its null-hypothesis variant, where every test faces a model that is right, and compares the two envelopes.

In [8]:
# Open a C22 artifact and compare its envelope with the C01 one
null = read_json("C22/null.json")
print("same envelope as C01:", list(null) == list(holdout))
print("truth status:", null["provenance"]["truth_status"], "| generators:", null["provenance"]["generators"])
print("C22 outputs keys:", list(null["outputs"]))
print("C01 outputs keys:", list(holdout["outputs"]))
print("test rows:", len(null["tests"]), "| first:", null["tests"][0]["test_id"], "on", null["tests"][0]["model_id"])

same envelope as C01: True
truth status: synthetic-known-truth | generators: ['binormal_scores', 'creditrisk_plus_sample', 'default_counts', 'downturn_lgd', 'es_losses', 'es_residuals', 'grade_frequencies', 'lgd_calibration', 'lhp_loss_rates', 'logistic_portfolio', 'mc_quantile', 'migrations', 'ml_covariate_shift_plant', 'ml_exchangeability_break_plant', 'pfe_series', 'pit_series', 'rating_paths', 'stage_transitions', 'threshold_bias_plant', 'two_sample', 'var_hits']
C22 outputs keys: ['kind', 'family', 'ladder', 'levels', 'panels', 'simulations', 'generators', 'golden', 'exact_curves', 'estimators', 'specimen', 'size_bounds']
C01 outputs keys: ['rungs', 'irb', 'n', 'defaults', 'groups', 'sample']
test rows: 5 | first: pd.hosmer_lemeshow on logistic-5000


## Results

To close the loop, the next cell lists the portfolio-level test rows of the C01 scorecard as the artifact stores them: which test, which value and which light. These are the numbers the later notebooks recompute with the installed engine.

In [9]:
# List the committed portfolio-level rows of the P1 scorecard: value (p-value when the test has one, else the metric) and light
rows = [row for row in holdout["tests"] if row["model_id"] == "P1-scorecard" and row["segment"] == "portfolio"]
for row in rows:
    value = row["p_value"] if row["p_value"] is not None else row["metric"]
    shown = "none" if value is None else format(value, ".6g")
    kind = "p-value" if row["p_value"] is not None else "metric"
    print(f'{row["test_id"]:22s} {kind:8s} {shown:>12s}  {row["light"]}')

disc.auc               metric       0.758722  not_evaluated
disc.auc_vs_initial    p-value       0.22574  green
disc.ks                p-value  8.14171e-220  not_evaluated
pd.jeffreys            p-value      0.309521  green
pd.binomial            p-value      0.314017  green
pd.binomial_vasicek    p-value      0.454241  green
pd.chi2_grades         p-value     0.0264445  amber
pd.hosmer_lemeshow     p-value     0.0150003  amber
pd.spiegelhalter       p-value     0.0218585  amber
pd.brier               metric       0.139272  not_evaluated
pd.ece                 metric      0.0155007  not_evaluated
stability.psi          p-value      0.781227  green
rating.hhi             p-value      0.497263  green


Reading the table. Most rows carry a p-value and a light (the level and fit tests). A few show the light `not_evaluated`: the AUC, the Brier score and the ECE are descriptive metrics, the Kolmogorov-Smirnov statistic is a measure of separation, and the default policy holds no threshold for any of them. The light is therefore the policy's reading of a p-value, never a property of the model on its own; notebook 03 of this topic shows how the policy turns a p-value into a colour. Each row also names its null hypothesis and its source, so a number can always be traced to the definition that produced it.

What this table does not show: it is one model (the scorecard) on one population (the 9,000 held-out cards), and it was computed by the engine release named in the provenance, not necessarily the one you installed.

## Summary

- `riskvalidation` 0.8.1 is installed from the pins of `notebooks/requirements.txt`, and a notebook prints the versions it runs.
- The engine's catalogue is the list of what a validator can run; each entry carries its null hypothesis and its source.
- Contraste commits three kinds of document: the index, a manifest per case and an artifact per variant, with one envelope for every case.
- Every artifact states its provenance, including the engine version that baked it, and a notebook compares that version with the installed one before it claims two numbers agree.

**What to try next.** Pick another artifact of C01 (for instance `data/derived/C01/drift-severe.json`) and compare its lights with the holdout's. Then continue with notebook 02, in which you call one test yourself on a real slice of 50 cards.

## References

- ECB (2019), Instructions for reporting the validation results of internal models: IRB Pillar I models for credit risk. https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf
- Yeh, I-C. (2009), Default of Credit Card Clients, UCI Machine Learning Repository (CC BY 4.0). https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
- The engine's catalogue and the document contracts: `docs/contracts/engine/` and `docs/contracts/` of this repository.